Last run: 260426

Uses results from astroquery.xmatch because it has the most (conservative) matches compared to exofop or vizier cross-match methods. See details in `crossmatch_TOI_with_exofop_vizier.ipynb`.

In [ ]:
import sys

!which {sys.executable}

In [ ]:
import sys
sys.path.insert(0, '..')

In [ ]:
import numpy as np
import pandas as pd

## TOI

In [ ]:
from wakai.catalog import get_tois

tois = get_tois(clobber=False) #ignore Gaia_exofop & Gaia_vizier
tois.shape

In [ ]:
tois.tail()

In [ ]:
tois = pd.read_csv('../data/TOIs_GaiaDR3_xmatch_id.csv', 
                   dtype={'GaiaDR3': 'Int64'})
tois.shape

In [ ]:
tois.columns

# Perren+2023 catalog

https://ucc.ar/changelog/

* [version 260423](https://zenodo.org/records/19712053)

Manual download from zenodo and upload to server

In [ ]:
import pandas as pd

df = pd.read_csv('../data/Perren2023/UCC_cat.csv')
df.tail()

In [ ]:
df = df.rename({"name": "Name",
                #"Source": "GaiaDR3",
                "probs": "Prob",
                "RA_ICRS": "ra",
                "DE_ICRS": "dec",
                "pmRA": "pmra",
                "pmDE": "pmdec",
                "Plx": "parallax",
                "RV": "radial_velocity"
                }, axis=1)
df

In [ ]:
df_mem = pd.read_parquet('../data/Perren2023/UCC_members.parquet')
df_mem.tail()

In [ ]:
df_mem.shape

In [ ]:
df_mem.Source.isin([2007818737548134912]).sum()

In [ ]:
#replace name to be compatible with other catalogs' naming formats
df_mem = df_mem.rename({"name": "Name",
                        "Source": "GaiaDR3",
                        "probs": "Prob",
                        "RA_ICRS": "ra",
                        "DE_ICRS": "dec",
                        "pmRA": "pmra",
                        "pmDE": "pmdec",
                        "Plx": "parallax",
                        "RV": "radial_velocity"
                        }, axis=1)
df_mem

In [ ]:
idx = df_mem.GaiaDR3.isin(tois.GaiaDR3)
idx.sum()

In [ ]:
df_mem[idx].shape

In [ ]:
idx = tois.GaiaDR3.isin(df_mem.GaiaDR3)
tois[idx].tail()

## match by 3-,2-D separation with cluster member
* 3d match yields lower number of matches since distance/parallax is taken into account in matching
* some of parallax is negative; such are treated as nan

In [ ]:
df_mem.parallax.shape, df_mem.parallax.dropna().shape

In [ ]:
(df_mem.parallax<0).sum()

In [ ]:
from astropy.coordinates import SkyCoord

# epoch already in J2016.0
toi_coords = SkyCoord(tois['RA'].values, tois['Dec'].values, 
                      distance=tois['Stellar Distance (pc)'].values, unit=('hourangle','deg','pc')) 
toi_coords

In [ ]:
from astropy.coordinates import SkyCoord, Distance
import astropy.units as u

coords1 = SkyCoord(ra=toi_coords.ra, dec=toi_coords.dec, distance=toi_coords.distance, frame='icrs')
coords2 = SkyCoord(ra=df_mem['ra'].values*u.degree, 
                   dec=df_mem['dec'].values*u.degree, 
                   distance=Distance(parallax=df_mem['parallax'].values* u.mas, allow_negative=True),
                   frame='icrs') 

# `match_to_catalog_sky` finds the nearest match in coords2 for each coordinate in coords1
# It returns the indices into the second catalog, the angular separation, and 3D separation (if applicable)
idx, d2d, d3d = coords1.match_to_catalog_sky(coords2)

# Add separation distance to df1
tois['sep2d_arcsec'] = d2d.to(u.arcsec).value
tois['sep3d_pc'] = d3d.to(u.pc).value
tois.sort_values('sep2d_arcsec').head()

In [ ]:
sep_2d = 3.0 * u.arcsec
sep_3d = 10 * u.pc #

# Add matching information to the original dataframes
match_mask_2d = d2d < sep_2d 
match_mask_3d = d3d < sep_3d

tois[match_mask_2d].shape, tois[match_mask_3d].shape

In [ ]:
tois[match_mask_3d][tois[match_mask_3d].sep3d_pc<1]

## What is the typical size of a cluster?

In [ ]:
df_mem.Name.unique().shape

In [ ]:
from wakai.catalog import get_cluster_3d_size

# test on praesepe
d = df_mem.groupby('Name').get_group('ngc2632')
get_cluster_3d_size(d)

In [ ]:
# apply func to all clusters
df_size = df_mem.groupby('Name').apply(get_cluster_3d_size)

In [ ]:
df = df.merge(df_size.reset_index(name='extent_pc'), on='Name')
df.head()

In [ ]:
df.extent_pc.describe()

## match by 3d separation with cluster core

In [ ]:
(df['parallax']<0).sum()

In [ ]:
from astropy.coordinates import SkyCoord, Distance

coords1 = SkyCoord(ra=toi_coords.ra, dec=toi_coords.dec, distance=toi_coords.distance, frame='icrs')
coords2 = SkyCoord(ra=df['ra'].values*u.degree, 
                   dec=df['dec'].values*u.degree, 
                   distance=Distance(parallax=df['parallax'].values* u.mas, allow_negative=True),
                   frame='icrs') 

idx, d2d, d3d = coords1.match_to_catalog_sky(coords2)

tois['sep2d_arcsec'] = d2d.to(u.arcsec).value 
tois['sep3d_pc'] = d3d.to(u.pc).value
tois.sort_values('sep2d_arcsec').head()

In [ ]:
tois.columns

In [ ]:
# align the two dfs
tois.shape, df.iloc[idx].shape

In [ ]:
# Add a prefix to matched columns to avoid confusion (optional but recommended)
col_prefix = 'matched_'
df_matched_toi = df.iloc[idx].copy().add_prefix(col_prefix)

# Concatenate horizontally
df_match = pd.concat([tois.reset_index(drop=True), 
                      df_matched_toi.reset_index(drop=True)
                      ], axis=1)

In [ ]:
# make sure the 3d sep from cluster core is smaller the spatial extent of that cluster
idx2 = df_match[f'{col_prefix}extent_pc'] > df_match['sep3d_pc']
df_match[idx2]

In [ ]:
import numpy as np

df_match['pm_diff'] =  np.sqrt((df_match[f'{col_prefix}pmra'] - df_match['PM RA (mas/yr)'])**2 + 
                               (df_match[f'{col_prefix}pmdec'] - df_match['PM Dec (mas/yr)'])**2)

In [ ]:
idx3 = df_match[idx2]['pm_diff'] < 5
cols = ['TOI','matched_Name','matched_Age_[Myr]',
        'matched_extent_pc','sep3d_pc','pm_diff','TIC ID',#'GaiaDR3_exofop','GaiaDR3_vizier'
       ]
df_final = df_match[idx2].loc[idx3,cols].reset_index(drop=True)
df_final

In [ ]:
#replace name to be compatible with other catalogs' naming formats
df_mem = df_mem.rename({#"name": "Name",
                        "GaiaDR3": "source_id",
                        #"probs": "Prob",
                        "RA_ICRS": "ra",
                        "DE_ICRS": "dec",
                        "pmRA": "pmra",
                        "pmDE": "pmdec",
                        "Plx": "parallax",
                        "RV": "radial_velocity"
                        }, axis=1)
df_mem.shape

In [ ]:
uniq = df_mem[df_mem.Name.isin(df_final.matched_Name.unique())]
counts = uniq.groupby("Name").count()
counts[counts.radial_velocity>10].sort_values('radial_velocity', ascending=False)

In [ ]:
df_final[df_final.matched_Name=='ngc2548']

In [ ]:
df_final[df_final.TOI==496.01]

In [ ]:
# target is excluded
df_final.query("TOI==6557.01")

In [ ]:
# target is excluded because extent_pc !> sep3d_pc
df_match.query("TOI==6557.01")[['TOI','matched_Name','matched_extent_pc','sep3d_pc','sep2d_arcsec','pm_diff']]

In [ ]:
target_gaiaid = 6000286264203417600
nearby_gaiaid = 6000286264212057344

In [ ]:
# not a cluster member
df_mem.query("source_id==@target_gaiaid")

In [ ]:
from wakai.catalog import plot_rdp_pmrv

toiid = 6557.01
df_target = df_match.query("TOI==@toiid").squeeze()
age = df_target["matched_Age_[Myr]"]
cname = df_target["matched_Name"]
d = df_mem.groupby("Name").get_group(cname)

fig = plot_rdp_pmrv(d,
                   color='Prob', 
                    target_gaiaid=6000286264203417600,
                    match_id=False,
                   df_target=df_target,
                   target_color="C3",
                   marker='.',
                   cmap="Blues");
fig.suptitle(f"TOI-{int(toiid)} in {cname}")

In [ ]:
get_cluster_3d_size(d)

In [ ]:
import matplotlib.pyplot as pl

ax = d.plot.scatter(x='ra',y='dec')
ax.plot(df.query("Name==@cname").squeeze().ra, df.query("Name==@cname").squeeze().dec, 'rx')
ax.plot(df_target.ra, df_target.dec, 'k+')

https://exofop.ipac.caltech.edu/tess/nearbytarget.php?id=141074654

In [ ]:
from wakai.catalog import plot_xyz_uvw

fig, ax = plot_xyz_uvw(d, 
                       color='Prob', 
                       df_target=df_target,
                       # target_gaiaid=target_gaiaid,
                       match_id=True,
                       target_color="C3",
                       target_ms=20,
                       marker='.',
                       cmap="Blues");
fig.suptitle(f"TOI-{int(toiid)} in {cname} ({age:.0f} Myr)")

In [ ]:
df_match['source_id'] = df_match.GaiaDR3_exofop.copy()

In [ ]:
import matplotlib.pyplot as pl
from wakai.catalog import plot_rdp_pmrv
from wakai.catalog import plot_xyz_uvw
from pathlib import Path

outdir = '../output/toi_near_clusters'
overwrite = False
groups = df_mem.groupby("Name")
for i,row in df_final.sort_values('TOI', ascending=False).iterrows():
    try:
        toi = row["TOI"]
        toiid = str(int(toi)).zfill(4)
        df_target = df_match.query("TOI==@toi").squeeze()
        cname = row["matched_Name"]
        age = row["matched_Age_[Myr]"]
        age_label = f"{age:.0f} Myr" if pd.notna(age) else "age unknown"
        d = groups.get_group(cname)
        if len(d.radial_velocity.dropna())<10:
            print(f"Skipping TOI {toiid}")
            continue
        # Cache key is TOI+cluster only (not age): re-running against updated
        # source data can shift a cluster's age estimate, and baking age into
        # the filename left stale duplicate PNGs behind under a different
        # name instead of the old one being replaced.
        fp = f'{outdir}/TOI-{toiid}_in_{cname}'

        if not Path(fp+'.png').exists() or overwrite:
            fig = plot_rdp_pmrv(d,
                               color='Prob',
                               match_id=False,
                               df_target=df_target,
                               target_color="C3",
                               marker='.',
                               cmap="Blues");
            fig.suptitle(f"TOI-{toiid} in {cname} ({age_label})")
            fig.savefig(fp+'.png', bbox_inches='tight', dpi=200)
        if not Path(fp+'_xyzuvw.png').exists() or overwrite:
            fig, ax = plot_xyz_uvw(d,
                               color='Prob',
                               df_target=df_target,
                               target_color="C3",
                               target_ms=20,
                               marker='.',
                               cmap="Blues",
                               verbose=False
                              );
            fig.suptitle(f"TOI-{toiid} in {cname} ({age_label})")
            fig.savefig(fp+'_xyzuvw.png', bbox_inches='tight', dpi=200)
        # pl.show()
        print(f"https://ucc.ar/_clusters/{cname}")
    except Exception as e:
        print(e)

In [ ]:
import subprocess
import glob

# Get all PNG files in the directory
images = glob.glob(f"{outdir}/*Myr.png")

# Run the command securely using subprocess
subprocess.run(["img2pdf"] + sorted(images) + ["-o", f"{outdir}.pdf"])

In [ ]:
images = glob.glob(f"{outdir}/*uvw.png")

# Run the command securely using subprocess
subprocess.run(["img2pdf"] + sorted(images) + ["-o", f"{outdir}_xyzuvw.pdf"])

In [ ]:
selected_toiids = [
    7761, #best
    #7577, #unlikely
    7497, #likely
    #7214, #unlikely
    7111, #maybe?
    6754, #maybe
    6462, #maybe
    6248, #likely
    6009, #maybe
    5973, #best
    5929, #maybe
    5244, #best
    5137, #good
    5039, #maybe
    4761, #maybe
    4510, #best
    4152, #maybe
    2048, #best
    1641, #
]

In [ ]:
idx = tois.TOI.astype(int).isin(selected_toiids)
cols = ['TOI', 'Depth (ppm)','Planet Radius (R_Earth)','Stellar Radius (R_Sun)','Stellar Eff Temp (K)',
        'Period (days)','Duration (hours)','TESS Mag','Sectors','Comments'
       ]
tois.loc[idx,cols].sort_values(by='TOI', ascending=False)

In [ ]:
!ls /ut2/jerome/github/research/project/tql-gallery/gallery/TOI/all/qlp/TOI6248*

# cross-match TOI with Gaia DR3 using `astroquery.xmatch`

In [ ]:
import pandas as pd

df = pd.read_csv('../data/Perren2023/UCC_cat.csv')
df.tail()

In [ ]:
df = df.rename({"name": "Name",
                #"Source": "GaiaDR3",
                "probs": "Prob",
                "RA_ICRS": "ra",
                "DE_ICRS": "dec",
                "pmRA": "pmra",
                "pmDE": "pmdec",
                "Plx": "parallax",
                "RV": "radial_velocity"
                }, axis=1)
df

In [ ]:
df_mem = pd.read_parquet('../data/Perren2023/UCC_members.parquet')
df_mem.tail()

In [ ]:
#replace name to be compatible with other catalogs' naming formats
df_mem = df_mem.rename({"name": "Name",
                        "Source": "GaiaDR3",
                        "probs": "Prob",
                        "RA_ICRS": "ra",
                        "DE_ICRS": "dec",
                        "pmRA": "pmra",
                        "pmDE": "pmdec",
                        "Plx": "parallax",
                        "RV": "radial_velocity"
                        }, axis=1)
df_mem

In [ ]:
import pandas as pd

# output of wakai.crossmatch_toi --catalog gaiaDR3
matches = pd.read_csv('../data/TOIs_GaiaDR3_xmatch_id.csv', 
                      dtype={'GaiaDR3': 'Int64'})
matches.tail()

In [ ]:
matches.GaiaDR3.unique().shape

In [ ]:
idx = matches.columns.isin(['ra','dec','pmra','pmdec','','radial_velocity','parallax'])
idx

In [ ]:
final = pd.merge(df_mem, matches[matches.columns[~idx]], on='GaiaDR3').sort_values(by='TOI', ascending=False)
final.head()

In [ ]:
final.shape

In [ ]:
desired_cols = "TOI	TIC ID	Name	GaiaDR3	Prob	TESS Disposition	TFOPWG Disposition	Comments	TESS Mag	TESS Mag err	Source_1	Detection	Epoch (BJD)	Epoch (BJD) err	Period (days)	Period (days) err	Duration (hours)	Duration (hours) err	Depth (mmag)	Depth (mmag) err	Depth (ppm)	Depth (ppm) err	Planet Radius (R_Earth)	Planet Radius (R_Earth) err	Planet Insolation (Earth Flux)	Planet Equil Temp (K)	Planet SNR	Stellar Distance (pc)	Stellar Distance (pc) err	Stellar Eff Temp (K)	Stellar Eff Temp (K) err	Stellar log(g) (cm/s^2)	Stellar log(g) (cm/s^2) err	Stellar Radius (R_Sun)	Stellar Radius (R_Sun) err	Stellar Metallicity	Stellar Metallicity err	Stellar Mass (M_Sun)	Stellar Mass (M_Sun) err	Sectors	Date TOI Alerted (UTC)	Date TOI Updated (UTC)	Date Modified	ra_deg	dec_deg	Pipeline Signal ID	RA	Dec	PM RA (mas/yr)	PM RA err (mas/yr)	PM Dec (mas/yr)	PM Dec err (mas/yr)	DR3Name	RAdeg	DEdeg	errHalfMaj	errHalfMin	errPosAng	SolID	Source_2	RandomI	e_RAdeg	e_DEdeg	RPlx	PM	RADEcor	RAPlxcor	RApmRAcor	RApmDEcor	DEPlxcor	DEpmRAcor	DEpmDEcor	PlxpmRAcor	PlxpmDEcor	pmRApmDEcor	NAL	NAC	NgAL	NbAL	gofAL	chi2AL	epsi	sepsi	Solved	APF	nueff	MatchObsA	Nper	amax	MatchObs	IPDgofha	IPDgofhp	IPDfmp	IPDfow	RUWE	Dup	o_Gmag	FG	e_FG	RFG	Gmag_y	o_BPmag	FBP	e_FBP	RFBP	e_BPmag	o_RPmag	FRP	e_FRP	RFRP	e_RPmag	E(BP/RP)	NBPcont	NBPblend	NRPcont	NRPblend	Mode	BP-G	G-RP	n_RV	o_RV	o_RVd	RVNper	RVS/N	RVgof	RVchi2	RVTdur	RVamp	RVtempTeff	RVtemplogg	RVtemp[Fe/H]	Vatmparam	vbroad	e_Vbroad	o_Vbroad	GRVSmag	e_GRVSmag	o_GRVSmag	RVSS/N	VarFlag	PQSO	PGal	PSS	Teff	b_Teff	B_Teff	logg	b_logg	B_logg	[Fe/H]	b_[Fe/H]	B_[Fe/H]	Dist	b_Dist	B_Dist	A0	b_A0	B_A0	AG	b_AG	B_AG	E(BP-RP)	b_E(BP-RP)	B_E(BP-RP)	Lib	RAJ2000	DEJ2000	e_RAJ2000	e_DEJ2000	RADEcorJ2000	Previous CTOI	Master	SG1A	SG1B	SG2	SG3	SG4	SG5	ESM	TSM	Predicted Mass (M_Earth)	Time Series Observations	Spectroscopy Observations	Imaging Observations".split('\t')
new_cols = [c for c in desired_cols if c in final.columns]
missing_cols = [c for c in desired_cols if c not in final.columns]
if missing_cols:
    print(f"Dropping {len(missing_cols)} columns not present in `final`: {missing_cols}")

In [ ]:
cnames = final.Name.unique()
cnames

In [ ]:
final2 = pd.merge(df, final, on='Name').sort_values(by='Age_[Myr]', ascending=True)

In [ ]:
idx = final2.Name.isin(cnames)
final2.loc[idx,['TOI','Name','Age_[Myr]','Prob']].drop_duplicates().reset_index(drop=True).sort_values(by='TOI')

In [ ]:
df_hosts = pd.read_csv("/ut2/jerome/github/research/project/TOI-6715/data/young_validated_planets_TOIs - validated_hosts.csv", 
                        skiprows=[1], comment='#').dropna(subset=['Host']).reset_index(drop=True)
# df_hosts = df_hosts[df_hosts.distance<500].reset_index(drop=True)
df_hosts.shape

In [ ]:
host_toiids = df_hosts.TOI.str.replace(r'\D', '', regex=True).replace('', np.nan).dropna().astype(int).unique()
host_toiids

In [ ]:
idx = final2.TOI.astype(int).isin(host_toiids)
final2.loc[~idx,['TOI','Name','Age_[Myr]','Prob']].drop_duplicates().reset_index(drop=True)

In [ ]:
df_old = pd.read_csv('../data/Perren2023/TOIs_in_Perren2023.csv')
df_old.shape

In [ ]:
final[new_cols].to_csv('../data/Perren2023/TOIs_in_Perren2023.csv', index=False)

In [ ]:
final[final['TIC ID']==64837857][['TOI','Period (days)','Epoch (BJD)', 'Duration (hours)']]

## distance

In [ ]:
import sys
sys.path.insert(0, '..')

In [ ]:
from wakai.catalog import get_tois

tois = get_tois()
tois = tois.dropna(subset=['ra_deg','dec_deg','Stellar Distance (pc)'])
tois.shape

In [ ]:
df = df.dropna(subset=['ra','dec','parallax'])[df.parallax>0]
df.shape

In [ ]:
df_mem = df_mem.dropna(subset=['ra','dec','parallax'])[df_mem.parallax>0]
df_mem.shape

In [ ]:
from astropy.coordinates import SkyCoord, Distance
import astropy.units as u

c1 = SkyCoord(ra=df.ra.values*u.deg, 
             dec=df.dec.values*u.deg,
             distance=Distance(parallax=df.parallax.values*u.mas)
             )
c2 = SkyCoord(ra=tois.ra_deg.values*u.deg, 
             dec=tois.dec_deg.values*u.deg,
             distance=tois['Stellar Distance (pc)'].values*u.pc
             )

sep = 20 #pc
use_2d = False
distances = []
idx = []
for c in c2:
    d = c1.separation(c).deg if use_2d else c1.separation_3d(c).pc
    if (pd.Series(d)<sep).sum()>0:
        idx.append(True)
        distances.append(min(d))
    else:
        idx.append(False)
        distances.append(np.nan)

In [ ]:
tois2 = tois.loc[idx]
tois2.shape

In [ ]:
idx2 = []
for c in c1:
    d = c1.separation_2d(c) if use_2d else c1.separation_3d(c)
    if (pd.Series(d.pc)<distance).sum()>0:
        idx2.append(True)
    else:
        idx2.append(False)

In [ ]:
names = df[idx2].Name.apply(lambda x: str(x).lower().replace(' ','').split(';')).explode().unique()
len(names)

In [ ]:
idx=df_mem.Name.isin(names)
idx.sum()

In [ ]:
df_mem2 = df_mem[idx]
df_mem2.shape

## visualization

In [ ]:
import pandas as pd

cluster_tois = pd.read_csv('../data/Perren2023/TOIs_in_Perren2023.csv')
cluster_tois.shape

In [ ]:
cluster_names = cluster_tois['Name'].unique()
len(cluster_names)

In [ ]:
idx = df_mem.Name.isin(cluster_names)

In [ ]:
df_mem2 = df_mem[idx]
df_mem2.shape

In [ ]:
# %load /ut3/jerome/github/research/project/wakai/notebooks/init_ipynb
import os
import sys
import matplotlib.pyplot as pl

# sys.path.insert(0, '../code')
# assert os.environ["CONDA_DEFAULT_ENV"] == 'wakai'

plotpar = {
           'font.size': 20,
           'xtick.labelsize': 16,
           'ytick.labelsize': 16,
           'xtick.direction': 'in',
           'ytick.direction': 'in',
           'xtick.color':'black',
           'ytick.color':'black',
           'xtick.major.width':2,
           'ytick.major.width':2,
           'xtick.major.size':10,
           'ytick.major.size':10,
           'xtick.minor.width':1,
           'ytick.minor.width':1,
           'xtick.minor.size':6,
           'ytick.minor.size':6,
           'axes.labelsize': 16,
           'axes.labelcolor':'black',
           'axes.labelcolor':'black',
           'axes.spines.top':True,
           'axes.spines.right':True,
           'axes.linewidth':2,
           'axes.edgecolor':'black',
           'figure.facecolor':'none',
           'legend.facecolor':'none',
           'legend.fontsize': 16,
           'text.color':'black',
           'pdf.fonttype': 42,
           #'text.usetex': False,
           }
pl.rcParams.update(plotpar)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u
from matplotlib.lines import Line2D 

def plot_cluster_density(df, cluster_tois=None, prob_threshold=0.5, 
                         bandwidth=0.3, min_members=5,plot_all_toi=False):
    """
    Plot 2D KDE contours for stellar clusters in Galactic coordinates.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing at least:
        ['Name', 'GLON', 'GLAT', 'Prob'] columns.
    prob_threshold : float, optional
        Minimum membership probability for inclusion.
    bandwidth : float, optional
        KDE bandwidth adjustment for smoothing.
    min_members : int, optional
        Minimum number of stars required to plot a cluster.
    """
    
    # Validate required columns
    x, y = 'ra', 'dec'
    p = 'Prob'
    n = 'Name'
    required_cols = {n, x, y, p}
    if not required_cols.issubset(df.columns):
        missing = required_cols - set(df.columns)
        raise ValueError(f"Missing required columns: {missing}")
    
    # Filter by membership probability
    filtered = df[df[p] >= prob_threshold].copy()
    if filtered.empty:
        raise ValueError("No stars remain after applying probability threshold.")

    custom_handles = []
    colors = plt.cm.tab10.colors # Use a color map for different clusters
    
    # Prepare plot
    fig, ax = plt.subplots(figsize=(10, 6), tight_layout=True)
    
    # Loop through clusters
    for i,(name, group) in enumerate(filtered.groupby(n)):
        if len(group) < min_members:
            continue
        #label = f"{name} (N={len(group)})"
        label = 'cluster' if i==0 else '_nolegend_'
        cluster_color = colors[i % len(colors)]
        sb.kdeplot(
            x=group[x],
            y=group[y],
            levels=3,
            fill=False,
            bw_adjust=bandwidth,
            linewidths=2,
            zorder=2,
            color=cluster_color,
            ax=ax
        )
        custom_handles.append(Line2D([0], [0], color=cluster_color, lw=2, label=label))
    ylims = ax.get_ylim()
    xlims = ax.get_xlim()
    if cluster_tois is not None:        
        s_toi = ax.scatter(cluster_tois['ra_deg'], cluster_tois['dec_deg'], marker='*',  
                   s=100, edgecolors='k', c='y', zorder=100, label='cluster TOI')
        ax.set_xlim(*xlims)
        ax.set_ylim(*ylims)
        custom_handles.append(s_toi)
    if plot_all_toi:
        s_all = ax.scatter(tois['ra_deg'], tois['dec_deg'], marker='.', c='k', alpha=0.1, zorder=1, label='TOI')
        custom_handles.append(s_all)
    ax.grid(alpha=0)
    ax.legend(handles=custom_handles, frameon=True, framealpha=1, facecolor="white", fontsize=16, loc='lower right')
    return ax

In [ ]:
ax = plot_cluster_density(df_mem2, 
                          cluster_tois,
                          prob_threshold=0.1, 
                          bandwidth=1, 
                          min_members=10,
                          plot_all_toi=True
                         )
ax.set_xlabel('R.A. [deg]')
ax.set_ylabel('Dec. [deg]')
ax.figure.savefig('TOI_in_Perren2023.pdf', bbox_inches='tight', dpi=200)